In [ ]:
### Load libraries and set params
library(Matrix)
library(parallel)
library(Seurat)
library(gridExtra)
library(ggplot2)
library(future)
library(ggpubr)
library(viridis)
library(openxlsx)
library(viridis)
library(scran)
library(gghighlight)
library(dplyr)
library(org.Dm.eg.db)
library(ggExtra)
library(scDblFinder)
library(patchwork)
library(RhpcBLASctl)
blas_set_num_threads(18)
library(peakRAM)
options(device=pdf)
options(future.globals.maxSize = 214748364800)
library(future)
plan("multicore", workers = 18)

### Set directories
mainDir <- "/data/ebaird/scRNAseq/SCENTINELsep24/"
repDir <- paste0(mainDir, "monocle3/")
figDir <- paste0(repDir, "figs/")
tabDir <- paste0(repDir, "tables/")
refsDir <- paste0(mainDir, "refs/")


dir.create(repDir, recursive = TRUE, showWarnings = FALSE)
dir.create(figDir, recursive = TRUE, showWarnings = FALSE)
dir.create(tabDir, recursive = TRUE, showWarnings = FALSE)

### Set colours
mycols <- c(1, '#ffffe5','#fff7bc','#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506')
mycols11 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray")
mycols13 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green")
mycols17 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green", rainbow(4))

mycols20 <- c("yellow", '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "chartreuse", "blue", "green", rainbow(4), "darkslategray3", "darksalmon", "darkorchid4", "cyan")

corner <- function(x) x[1:5,1:5]
cols <- c(colorRamps::matlab.like2(20)[1:18], "deeppink2", "deeppink3", "deeppink4")

getdensity <- function(x, y, ...) {
      dens <- MASS::kde2d(x, y, ...)
      ix <- findInterval(x, dens$x)
      iy <- findInterval(y, dens$y)
      ii <- cbind(ix, iy)
      return(dens$z[ii])
}

In [ ]:
library(lme4)
library(monocle3, lib.loc = "/data/ebaird/miniconda3/envs/r_monocle/lib/R/library")
library(SeuratWrappers, lib.loc = "/data/ebaird/miniconda3/envs/r_monocle/lib/R/library")

seurat_path <- file.path(mainDir, "subset_reanalysis/neuronal_subset.rds")
seu <- readRDS(seurat_path)
cds <- as.cell_data_set(seu)
cds <- preprocess_cds(cds, num_dim = 50)
cds <- cluster_cells(cds, reduction_method = "UMAP")
cds <- learn_graph(cds)
colnames(colData(cds))

In [ ]:
plot_cells(cds,
           color_cells_by = "seurat_clusters",
           label_groups_by_cluster=FALSE,
           label_leaves=FALSE,
           label_branch_points=FALSE)

In [ ]:
# compute root node
get_earliest_principal_node <- function(cds, time_bin="0"){
  cell_ids <- which(colData(cds)[, "seurat_clusters"] == time_bin)
  
  closest_vertex <-
  cds@principal_graph_aux[["UMAP"]]$pr_graph_cell_proj_closest_vertex
  closest_vertex <- as.matrix(closest_vertex[colnames(cds), ])
  root_pr_nodes <-
  igraph::V(principal_graph(cds)[["UMAP"]])$name[as.numeric(names
  (which.max(table(closest_vertex[cell_ids,]))))]
  
  root_pr_nodes
}
cds <- order_cells(cds, root_pr_nodes=get_earliest_principal_node(cds, time_bin="0"))

jpeg(file.path(figDir, paste0("trajectory_plot_neuronal_umap.jpeg")), width = 1000, height = 1000, res = 150)
plot_cells(cds, color_cells_by = "pseudotime", label_cell_groups = FALSE, label_leaves = FALSE, label_branch_points = FALSE, show_trajectory_graph = FALSE)
dev.off()

In [ ]:
library(spdep, lib.loc = "/data/ebaird/miniconda3/envs/r_monocle/lib/R/library")
library(sf, lib.loc = "/data/ebaird/miniconda3/envs/r_monocle/lib/R/library")
library(pbmcapply, lib.loc = "/data/ebaird/miniconda3/envs/r_monocle/lib/R/library")

cds_graph_test <- graph_test(cds, neighbor_graph="principal_graph", cores=4)
deg_ids <- row.names(subset(cds_graph_test, q_value < 0.05))

In [ ]:
write.csv(cds_graph_test, file=paste0(tabDir,'monocle3_DEG_neuronal_pseudotime.csv'))

In [ ]:
genes_to_plot <- c(
  "ase", "SoxN", "Eip93F", "Ank2", "sm", "Hsp26", "bi", "br",
  "E(spl)malpha-BFM", "E(spl)mbeta-HLH", "Sh", "Bacc",
  "wor", "Appl", "Myo81F", "Hsp23", "cpo", "HmgD",
  "nrv3", "Thor", "CG46301", "Syp", "sna", "nerfin-1",
  "Eip75B", "elav"
)


valid_genes <- intersect(genes_to_plot, rownames(rowData(cds)))
rowData(cds)$gene_short_name <- rownames(cds)


for (gene in valid_genes) {
  p <- plot_cells(cds,
                 genes = gene,
                 show_trajectory_graph = FALSE,
                 label_cell_groups = FALSE,
                 label_leaves = FALSE)
  print(p)  
}


In [ ]:
gene_module_df <- find_gene_modules(cds[deg_ids,], resolution=c(10^seq(-6,-1)))

In [ ]:
library(grr, lib.loc = "/data/ebaird/miniconda3/envs/r_monocle/lib/R/library")

cell_group_df <- tibble::tibble(cell=row.names(colData(cds)), 
                                cell_group=colData(cds)$seurat_clusters)
agg_mat <- aggregate_gene_expression(cds, gene_module_df, cell_group_df)
row.names(agg_mat) <- stringr::str_c("Module ", row.names(agg_mat))
pheatmap::pheatmap(agg_mat,
                   scale="column", clustering_method="ward.D2")

In [ ]:
gene_module_df %>% filter(module %in% 17)

In [ ]:
plot_cells(cds,
           genes=gene_module_df %>% filter(module %in% modules),
           label_cell_groups=FALSE,
           show_trajectory_graph=FALSE)

In [ ]:
### Gene dynamics over pseudotime

library(dplyr)
library(tibble)
library(tidyr)

pseudotime <- pseudotime(cds)
exprs_matrix <- SingleCellExperiment::counts(cds)

goi <- c("Thor", "nerfin-1")
exprs_subset <- exprs_matrix[goi, ]

df <- as.data.frame(t(exprs_subset))
df$pseudotime <- pseudotime

df_long <- df %>%
  pivot_longer(cols = -pseudotime, names_to = "gene", values_to = "expression")

if ("bin" %in% colnames(df_long)) {
  df_long <- df_long %>% select(-bin)
}

df_long <- df_long %>%
  mutate(bin = cut(pseudotime, breaks=25))

avg_exprs <- df_long %>%
    group_by(gene, bin) %>%
    summarise(
        avg_expression = mean(expression, na.rm=TRUE),
        pseudotime_bin = mean(pseudotime, na.rm=TRUE),
        .groups = 'drop'
    )

ggplot(avg_exprs, aes(x=pseudotime_bin, y=avg_expression, color=gene)) +
    geom_line(size=1) +
    labs(x = "Pseudotime", y = "Average expression", title = "Gene expression dynamics over pseudotime") +
    theme_classic()

In [ ]:
#########################
####STILL IN PROGRESS####
#########################
# 1. Create trajectory assignments preserving hierarchy
cds$trajectory <- case_when(
    cds$seurat_clusters == "3" ~ "Traj_A",
    cds$seurat_clusters == "4" ~ "Traj_C",
    cds$seurat_clusters == "5" ~ "Traj_B",
    TRUE ~ "Intermediate"  # Includes clusters 0, 1, 2
)

# 2. Define trajectory paths with explicit hierarchy
trajectories <- list(
    "Traj_A" = list(
        path = c("0", "2", "3"),
        intermediates = c("0", "2")
    ),
    "Traj_B" = list(
        path = c("0", "2", "1", "5"),
        intermediates = c("0", "2", "1")
    ),
    "Traj_C" = list(
        path = c("0", "2", "1", "4"),
        intermediates = c("0", "2", "1")
    )
)

# 3. Enhanced trajectory analysis function
trajectory_analysis <- function(traj_name, traj_def) {
    # Get all cells in path
    path_cells <- colnames(cds)[cds$seurat_clusters %in% traj_def$path]
    
    # Create subset
    cds_sub <- cds[, path_cells]
    
    # Recalculate pseudotime with correct root
    cds_sub <- preprocess_cds(cds_sub, num_dim = 50)
    cds_sub <- reduce_dimension(cds_sub)
    cds_sub <- cluster_cells(cds_sub)
    cds_sub <- learn_graph(cds_sub)
    
    # Set root to first cluster in path
    root_node <- get_earliest_principal_node(cds_sub, time_bin = traj_def$path[1])
    cds_sub <- order_cells(cds_sub, root_pr_nodes = root_node)
    
    # Perform DE analysis
    de_test <- graph_test(
        cds_sub,
        neighbor_graph = "principal_graph",
        cores = 18,
        expression_family = "negbinomial"
    )
    
    # Add metadata
    de_test$trajectory <- traj_name
    de_test$root_cluster <- traj_def$path[1]
    de_test$terminal_cluster <- tail(traj_def$path, 1)
    
    return(de_test)
}

# 4. Run analysis for each trajectory
de_results <- list()
for (traj_name in names(trajectories)) {
    de_results[[traj_name]] <- trajectory_analysis(traj_name, trajectories[[traj_name]])
}

# 5. Compare intermediate vs terminal states
terminal_de <- fit_models(
    cds[, cds$trajectory %in% c("Traj_A", "Traj_B", "Traj_C")],
    model_formula_str = "~trajectory",
    expression_family = "negbinomial",
    cores = 18
) %>%
    compare_models(reduced_model_formula_str = "~1") %>%
    filter(q_value < 0.05)

# 6. Save all results
all_de <- do.call(rbind, de_results)
write.csv(all_de, file.path(tabDir, "trajectory_specific_genes.csv"))
write.csv(terminal_de, file.path(tabDir, "terminal_state_genes.csv"))